In [1]:
!pip install pyyaml


  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl (156 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install pandas numpy streamlit 

  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached pandas-2.3.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached pillow-12.1.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached watchdog-6.0.0-py3-none-win_amd64.whl.metadata (44 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 k


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [66]:
import pandas as pd
import os
import yaml
import numpy as np

In [ ]:
# Data Extraction and Transformation: 
# ● Data is provided in YAML format, organized by months. 
# ● Within each month's folder, there are date-wise data entries. 
# ● The task is to extract this data from the YAML file and transform it into a CSV 
# format, organized by symbols  
# ● This will result in 50 CSV files after the extraction process, one for each symbol 
# or data category

In [ ]:
root_folder = r'C:\Nithyanantham\Mini_Projects\data'
output_dir = r'C:\Nithyanantham\Mini_Projects\extractdata'
date_folders=os.listdir(root_folder)
all_datas=[]
try:
    for i in date_folders:
        file_folders=os.listdir(os.path.join(root_folder,i))
        for j in file_folders:
            with open(os.path.join(root_folder, i,j), "r") as file:
                data = yaml.safe_load(file)
                all_datas.extend(data)
    df=pd.DataFrame(all_datas)
    for ticker, group in df.groupby('Ticker'):
        # with open(output_dir,'w') as file:
        file_path = os.path.join(output_dir, f"{ticker}.csv")
        group.to_csv(file_path, index=False)
except Exception as e:
    print(e)

In [ ]:
# Python DataFrame for Key Metrics: 
# ○ Market Summary: 
# ○ Top 10 Green Stocks: Sort the stocks based on their yearly return and select 
# the top 10. 
# ○ Top 10 Loss Stocks: Sort the stocks based on their yearly return and select 
# the bottom 10. 
# ■ Calculate the overall number of green vs. red stocks. 
# ■ Calculate the average price across all stocks. 
# ■ Calculate the average Volume across all stocks. 

In [30]:
dfs=[]
output_dir = r'C:\Nithyanantham\Mini_Projects\extractdata'
for file in os.listdir(output_dir):
    file_path = os.path.join(output_dir, file)
    data = pd.read_csv(file_path)
    dfs.append(data)
df = pd.concat(dfs, ignore_index=True)
df.shape

(14200, 8)

In [ ]:
# ○ Top 10 Green Stocks: Sort the stocks based on their yearly return and select
# the top 10.
# ○ Top 10 Loss Stocks: Sort the stocks based on their yearly return and select
# the bottom 10.

In [ ]:
df['date']=pd.to_datetime(df['date']) # Convert Object in to Date Type 

In [42]:
df['date'].dt.year

0        2023
1        2023
2        2023
3        2023
4        2023
         ... 
14195    2024
14196    2024
14197    2024
14198    2024
14199    2024
Name: date, Length: 14200, dtype: int32

In [56]:
df_one=df
df_one['financial_year'] = df_one['date'].apply(
    lambda x: x.year if x.month >= 4 else x.year - 1
)
df_one=df_one.groupby(['Ticker']).agg(first_close=('close', 'first'),last_close=('close', 'last')
).reset_index()
df_one['yearly_returns'] = (
    (df_one['last_close'] - df_one['first_close'])
    / df_one['first_close']
) * 100


In [ ]:
# ○ Top 10 Green Stocks: Sort the stocks based on their yearly return and select
# the top 10.

In [133]:
top10=df_one.sort_values('yearly_returns',ascending=False,ignore_index=True).head(10)
top10.to_csv(r'C:\Nithyanantham\Mini_Projects\results\Top 10 Green Stocks.csv',index=False)

In [134]:
bottom10 = df_one.sort_values(
    'yearly_returns', ascending=True, ignore_index=True).head(10)
bottom10

bottom10.to_csv(
    r'C:\Nithyanantham\Mini_Projects\results\Bottom 10 Red Stocks.csv', index=False)

In [ ]:
df_one['Stock_Status'] = np.where(df_one['yearly_returns'] >= 0, 'Green', 'Red')
df_one

In [140]:
# ■ Calculate the overall number of green vs. red stocks.
df_one.to_csv(r'C:\Nithyanantham\Mini_Projects\results\before_ques.csv',index=False)

In [137]:
# Group by Stock_Status and count
green_red_counts = df_one.groupby(
    'Stock_Status').size().reset_index(name='Count')

# Optional: Add percentage of each status
total = green_red_counts['Count'].sum()
green_red_counts['Percentage'] = (
    green_red_counts['Count'] / total * 100).round(2)

# Export to CSV
green_red_counts.to_csv(
    r"C:\Nithyanantham\Mini_Projects\results\green_vs_red_stocks.csv",
    index=False
)

print(green_red_counts)

  Stock_Status  Count  Percentage
0        Green     45     90.0000
1          Red      5     10.0000


In [ ]:
# ■ Calculate the average price across all stocks.

In [75]:
df_one['yearly_returns'].mean()

np.float64(32.84603162115448)

In [ ]:
# ■ Calculate the average Volume across all stocks.

In [77]:
df['volume'].mean()

np.float64(6833474.649154929)

In [82]:
df.drop('financial_year', axis=1, inplace=True)

In [83]:
df

,Ticker,close,date,high,low,month,open,volume
0,ADANIENT,2387.25,2023-10-03 05:30:00,2424.90,2372.00,2023-10,2418.00,2019899
1,ADANIENT,2464.95,2023-10-04 05:30:00,2502.75,2392.25,2023-10,2402.20,2857377
2,ADANIENT,2466.35,2023-10-05 05:30:00,2486.50,2446.40,2023-10,2477.95,1132455
3,ADANIENT,2478.10,2023-10-06 05:30:00,2514.95,2466.05,2023-10,2466.35,1510035
4,ADANIENT,2442.60,2023-10-09 05:30:00,2459.70,2411.30,2023-10,2440.00,1408224
...,...,...,...,...,...,...,...,...
14195,WIPRO,566.70,2024-11-14 05:30:00,574.55,564.20,2024-11,568.95,4891760
14196,WIPRO,552.85,2024-11-18 05:30:00,566.70,540.30,2024-11,566.70,7644882
14197,WIPRO,562.00,2024-11-19 05:30:00,569.80,554.70,2024-11,556.00,6459889
14198,WIPRO,557.15,2024-11-21 05:30:00,567.60,555.30,2024-11,562.00,5836304


In [85]:
df['daily_returns'] = df['close'].pct_change()
df['daily_returns_percent'] = df['close'].pct_change() * 100

In [ ]:
df = df.dropna(subset=['daily_returns'])
df 

,Ticker,close,date,high,low,month,open,volume,daily_returns,daily_returns_percent
1,ADANIENT,2464.95,2023-10-04 05:30:00,2502.75,2392.25,2023-10,2402.20,2857377,0.032548,3.254791
2,ADANIENT,2466.35,2023-10-05 05:30:00,2486.50,2446.40,2023-10,2477.95,1132455,0.000568,0.056796
3,ADANIENT,2478.10,2023-10-06 05:30:00,2514.95,2466.05,2023-10,2466.35,1510035,0.004764,0.476413
4,ADANIENT,2442.60,2023-10-09 05:30:00,2459.70,2411.30,2023-10,2440.00,1408224,-0.014325,-1.432549
5,ADANIENT,2498.30,2023-10-10 05:30:00,2517.95,2443.00,2023-10,2443.00,1771910,0.022804,2.280357
...,...,...,...,...,...,...,...,...,...,...
14195,WIPRO,566.70,2024-11-14 05:30:00,574.55,564.20,2024-11,568.95,4891760,-0.004042,-0.404218
14196,WIPRO,552.85,2024-11-18 05:30:00,566.70,540.30,2024-11,566.70,7644882,-0.024440,-2.443974
14197,WIPRO,562.00,2024-11-19 05:30:00,569.80,554.70,2024-11,556.00,6459889,0.016551,1.655060
14198,WIPRO,557.15,2024-11-21 05:30:00,567.60,555.30,2024-11,562.00,5836304,-0.008630,-0.862989


In [88]:
df['daily_returns'].mean()

np.float64(0.0047477803163692225)

In [93]:
df.groupby('Ticker')['daily_returns'].sum()

Ticker
ADANIENT       0.051582
ADANIPORTS    -0.215139
APOLLOHOSP     3.835057
ASIANPAINT    -0.768021
AXISBANK      -0.451532
BAJAJ-AUTO     4.072764
BAJAJFINSV    -0.782031
BAJFINANCE     3.837394
BEL           -0.197265
BHARTIARTL     2.849529
BPCL          -0.306104
BRITANNIA     14.826269
CIPLA         -0.490060
COALINDIA     -0.388338
DRREDDY        1.766159
EICHERMOT      2.193417
GRASIM        -0.280648
HCLTECH       -0.067353
HDFCBANK      -0.033378
HDFCLIFE      -0.524192
HEROMOTOCO     3.884372
HINDALCO      -0.538242
HINDUNILVR     2.796966
ICICIBANK     -0.285294
INDUSINDBK    -0.185770
INFY           0.748706
ITC           -0.672401
JSWSTEEL       0.898902
KOTAKBANK      0.815729
LT             0.945181
M&M            0.152054
MARUTI         2.527534
NESTLEIND     -0.769166
NTPC          -0.422201
ONGC          -0.138135
POWERGRID      0.386534
RELIANCE       2.553628
SBILIFE        0.192508
SBIN          -0.245040
SHRIRAMFIN     1.815444
SUNPHARMA     -0.126893
TATACONSU

In [ ]:
# 1. Volatility Analysis:  
# ● Objective: Visualize the volatility of each stock over the past year by calculating the 
# standard deviation of daily returns. 
# ● Reason: Volatility gives insight into how much the price fluctuates, which is valuable 
# for risk assessment. Higher volatility often indicates more risk, while lower volatility 
# indicates a more stable stock. 
# Metrics: 
# ● Calculate daily returns for each stock: (Close Price - Previous Close 
# Price) / Previous Close Price. 
# ● Compute the standard deviation of daily returns for each stock to measure volatility. 
# ● Plot a bar chart showing the volatility of the top 10 most volatile stocks over the year. 
# Visualization: 
# ● Top 10 Most Volatile Stocks: A bar chart with the stock ticker on the x-axis and 
# volatility (standard deviation) on the y-axis. 

In [94]:
df

,Ticker,close,date,high,low,month,open,volume,daily_returns,daily_returns_percent
1,ADANIENT,2464.95,2023-10-04 05:30:00,2502.75,2392.25,2023-10,2402.20,2857377,0.032548,3.254791
2,ADANIENT,2466.35,2023-10-05 05:30:00,2486.50,2446.40,2023-10,2477.95,1132455,0.000568,0.056796
3,ADANIENT,2478.10,2023-10-06 05:30:00,2514.95,2466.05,2023-10,2466.35,1510035,0.004764,0.476413
4,ADANIENT,2442.60,2023-10-09 05:30:00,2459.70,2411.30,2023-10,2440.00,1408224,-0.014325,-1.432549
5,ADANIENT,2498.30,2023-10-10 05:30:00,2517.95,2443.00,2023-10,2443.00,1771910,0.022804,2.280357
...,...,...,...,...,...,...,...,...,...,...
14195,WIPRO,566.70,2024-11-14 05:30:00,574.55,564.20,2024-11,568.95,4891760,-0.004042,-0.404218
14196,WIPRO,552.85,2024-11-18 05:30:00,566.70,540.30,2024-11,566.70,7644882,-0.024440,-2.443974
14197,WIPRO,562.00,2024-11-19 05:30:00,569.80,554.70,2024-11,556.00,6459889,0.016551,1.655060
14198,WIPRO,557.15,2024-11-21 05:30:00,567.60,555.30,2024-11,562.00,5836304,-0.008630,-0.862989


In [100]:
volatility = df.groupby('Ticker')['daily_returns'].std()
volatility.sort_values(ascending=False)

Ticker
TCS           1.401027
BRITANNIA     0.873939
BAJFINANCE    0.236561
APOLLOHOSP    0.208282
BAJAJ-AUTO    0.201852
HEROMOTOCO    0.201252
HINDUNILVR    0.165760
RELIANCE      0.145390
MARUTI        0.145067
BHARTIARTL    0.136724
EICHERMOT     0.105553
DRREDDY       0.097978
SHRIRAMFIN    0.083185
BEL           0.062732
WIPRO         0.059899
BPCL          0.057403
HINDALCO      0.056937
NTPC          0.056518
TATASTEEL     0.052993
COALINDIA     0.052337
BAJAJFINSV    0.051545
TITAN         0.051169
NESTLEIND     0.048996
CIPLA         0.047786
KOTAKBANK     0.047668
LT            0.047286
ITC           0.047173
ADANIPORTS    0.045443
TECHM         0.045334
HDFCLIFE      0.040548
JSWSTEEL      0.040359
SBIN          0.039602
GRASIM        0.039377
M&M           0.039188
ICICIBANK     0.038753
AXISBANK      0.037751
SUNPHARMA     0.037546
ONGC          0.036877
ASIANPAINT    0.034589
HCLTECH       0.034262
TATACONSUM    0.034253
TRENT         0.032319
INFY          0.029615
ADAN

In [142]:
annual_volatility = volatility * np.sqrt(252)
top10 = annual_volatility.sort_values(ascending=False).head(10)
top10.to_csv(r'C:\Nithyanantham\Mini_Projects\results\question1.csv',index=False)
top10

Ticker
TCS          22.2406
BRITANNIA    13.8734
BAJFINANCE    3.7553
APOLLOHOSP    3.3064
BAJAJ-AUTO    3.2043
HEROMOTOCO    3.1948
HINDUNILVR    2.6314
RELIANCE      2.3080
MARUTI        2.3029
BHARTIARTL    2.1704
Name: daily_returns, dtype: float64

In [143]:
top10_df = top10.reset_index()
top10_df.columns = ['Ticker', 'Cumulative_Return']

top10_df.to_csv(
    r'C:\Nithyanantham\Mini_Projects\results\question1.csv',
    index=False
)

In [ ]:
# 2. Cumulative Return Over Time: 
# ● Objective: Show the cumulative return of each stock from the beginning of the year to 
# the end. 
# ● Reason: The cumulative return is an important metric to visualize overall performance 
# and growth over time. This helps users compare how different stocks performed 
# relative to each other. 
# Metrics: 
# ● For each stock, calculate the cumulative return by applying a running total of daily 
# returns. 
# ● Plot a line chart for the top 5 performing stocks (based on cumulative return) over the 
# course of the year. 
# Visualization: 
# ● Cumulative Return for Top 5 Performing Stocks: A line chart displaying cumulative 
# returns for each stock over the year (increasing trend indicates positive performance).

In [106]:
df.groupby('Ticker')

In [122]:
day_zero_return=0
df['cumulative_return']=(1 + df['daily_returns']).groupby(df['Ticker']).cumprod() - 1

In [166]:
top5 = df.groupby('Ticker')['cumulative_return'].last()
top5 = top5.sort_values(ascending=False).head(5)
top5

Ticker
TCS          28.7283
BRITANNIA    15.9612
BAJAJ-AUTO    7.2998
HEROMOTOCO    5.9661
APOLLOHOSP    5.1008
Name: cumulative_return, dtype: float64

In [174]:
top5_tickers = (
    df.groupby('Ticker')['cumulative_return']
      .last()
      .sort_values(ascending=False)
      .head(5)
      .index
)

top5_tick = []

for ticker in top5_tickers:
    top5_tick.append(df[df['Ticker'] == ticker])

ans = pd.concat(top5_tick, ignore_index=True)
ans.to_csv(r'C:\Nithyanantham\Mini_Projects\results\question2.csv', index=False)

In [ ]:
# 3. Sector-wise Performance: 
# ● Objective: Provide a breakdown of stock performance by sector (sector data shared 
# as csv). 
# ● Reason: Investors and analysts often look at sector performance to gauge market 
# sentiment in specific industries (e.g., IT, Financials, Energy, etc.). 
# ● Classify each stock by its sector (this can be done by adding a separate dataset or 
# manually mapping sectors to stocks). 
# ● Calculate the average yearly return for each sector. 
# ● Plot a bar chart showing the average performance for each sector. 
# Visualization: 
# ● Average Yearly Return by Sector: A bar chart where each bar represents a sector 
# and its height indicates the average yearly return for stocks within that sector. 

In [144]:
sector_df=pd.read_csv('Sector_Mapped_DF.csv')
sector_df.head()

,Ticker,close,date,high,low,month,open,volume,Daily_Return,Cumulative_return,sector
0,ADANIENT,2464.9500,10/4/2023 5:30,2502.7500,2392.2500,2023-10,2402.2000,2857377,0.0325,0.0325,MINING
1,ADANIENT,2466.3500,10/5/2023 5:30,2486.5000,2446.4000,2023-10,2477.9500,1132455,0.0006,0.0331,MINING
2,ADANIENT,2478.1000,10/6/2023 5:30,2514.9500,2466.0500,2023-10,2466.3500,1510035,0.0048,0.0381,MINING
3,ADANIENT,2442.6000,10/9/2023 5:30,2459.7000,2411.3000,2023-10,2440.0000,1408224,-0.0143,0.0232,MINING
4,ADANIENT,2498.3000,10/10/2023 5:30,2517.9500,2443.0000,2023-10,2443.0000,1771910,0.0228,0.0465,MINING


In [159]:
sector_performance = sector_df.groupby('sector')['Daily_Return'].mean()
sector_performance*100

sector
ALUMINIUM             0.1278
AUTOMOBILES           0.1605
BANKING               0.0423
CEMENT                0.1214
DEFENCE               0.2763
ENERGY                0.1243
ENGINEERING           0.0708
FINANCE               0.0440
FMCG                  0.0038
FOOD                  0.0352
FOOD & TOBACCO        0.0222
FOOD AND BEVERAGES    0.0438
INSURANCE             0.0502
MINING                0.0825
MISCELLANEOUS         0.1314
PAINTS               -0.0794
PHARMACEUTICALS       0.1021
POWER                 0.1845
RETAILING             0.2315
SOFTWARE              0.1249
STEEL                 0.0764
TELECOM               0.1961
TEXTILES              0.1186
Name: Daily_Return, dtype: float64

In [160]:
sector_performance = (
    sector_df
    .groupby('sector', as_index=False)['Daily_Return']
    .mean()
)
sector_performance

,sector,Daily_Return
0,ALUMINIUM,0.0013
1,AUTOMOBILES,0.0016
2,BANKING,0.0004
3,CEMENT,0.0012
4,DEFENCE,0.0028
5,ENERGY,0.0012
6,ENGINEERING,0.0007
7,FINANCE,0.0004
8,FMCG,0.0000
9,FOOD,0.0004


In [161]:
sector_performance.to_csv(r'C:\Nithyanantham\Mini_Projects\results\question3.csv', index=False)

In [ ]:
# 4. Stock Price Correlation: 

# Objective: Visualize the correlation between the stock prices of different companies. 
 
# ● Reason: This analysis is valuable to understand if certain stocks tend to move in 
# tandem (e.g., correlated with market trends or sector performance). 
# Metrics: 
# ● Calculate the correlation coefficient between the closing percentage of different 
# stocks. For this, use the pandas.DataFrame.corr() method. 
# ● Create a correlation matrix to identify how stocks are related to each other. 
# ● Plot a heatmap of the correlation matrix to visualize these relationships. 
# Visualization: 
# ● Stock Price Correlation Heatmap: A heatmap to show the correlation between the 
# closing prices of various stocks. Darker colors represent higher correlations.

In [179]:
returns_pivot = df.pivot(
    index='date',
    columns='Ticker',
    values='daily_returns'
)
correlation_matrix = returns_pivot.corr()
correlation_matrix.to_csv(r'C:\Nithyanantham\Mini_Projects\results\question4.csv', index=False)

In [183]:
correlation_matrix = pd.DataFrame(
    correlation_matrix.values,
    index=returns_pivot.columns,
    columns=returns_pivot.columns
)
correlation_matrix

Ticker,ADANIENT,ADANIPORTS,APOLLOHOSP,ASIANPAINT,AXISBANK,BAJAJ-AUTO,BAJAJFINSV,BAJFINANCE,BEL,BHARTIARTL,...,SUNPHARMA,TATACONSUM,TATAMOTORS,TATASTEEL,TCS,TECHM,TITAN,TRENT,ULTRACEMCO,WIPRO
Ticker,,,,,,,,,,,,,,,,,,,,,
ADANIENT,1.0000,0.8742,0.1365,0.2916,0.2977,0.2019,0.3565,0.3360,0.5227,0.2829,...,0.1663,0.2038,0.3241,0.4236,0.1158,0.1417,0.2833,0.1904,0.3612,0.2240
ADANIPORTS,0.8742,1.0000,-0.8120,0.8209,0.8370,-0.8071,0.8514,-0.8033,0.8861,-0.7983,...,0.8180,0.7946,0.7457,0.8620,-0.8197,0.7975,-0.7419,0.6575,-0.4270,0.8247
APOLLOHOSP,0.1365,-0.8120,1.0000,-0.9223,-0.9035,0.9952,-0.9551,0.9965,-0.9211,0.9944,...,-0.9419,-0.8972,-0.7422,-0.9311,0.9977,-0.9295,0.9637,-0.6956,0.7275,-0.9501
ASIANPAINT,0.2916,0.8209,-0.9223,1.0000,0.8667,-0.9219,0.9201,-0.9214,0.8858,-0.9208,...,0.9074,0.8782,0.7267,0.9102,-0.9302,0.8920,-0.8637,0.7111,-0.5950,0.9055
AXISBANK,0.2977,0.8370,-0.9035,0.8667,1.0000,-0.9001,0.9204,-0.8988,0.8964,-0.8963,...,0.8838,0.8464,0.7492,0.9079,-0.9102,0.8720,-0.8587,0.6976,-0.5377,0.8962
BAJAJ-AUTO,0.2019,-0.8071,0.9952,-0.9219,-0.9001,1.0000,-0.9526,0.9950,-0.9178,0.9935,...,-0.9424,-0.8917,-0.7337,-0.9271,0.9962,-0.9310,0.9624,-0.6850,0.7295,-0.9505
BAJAJFINSV,0.3565,0.8514,-0.9551,0.9201,0.9204,-0.9526,1.0000,-0.9460,0.9285,-0.9501,...,0.9321,0.9037,0.7751,0.9463,-0.9613,0.9198,-0.8998,0.7140,-0.6005,0.9446
BAJFINANCE,0.3360,-0.8033,0.9965,-0.9214,-0.8988,0.9950,-0.9460,1.0000,-0.9189,0.9947,...,-0.9451,-0.8932,-0.7423,-0.9283,0.9978,-0.9295,0.9640,-0.6889,0.7349,-0.9499
BEL,0.5227,0.8861,-0.9211,0.8858,0.8964,-0.9178,0.9285,-0.9189,1.0000,-0.9128,...,0.9014,0.8660,0.7893,0.9386,-0.9284,0.8982,-0.8758,0.7211,-0.5526,0.9162


In [184]:
correlation_matrix.to_csv(
    r"C:\Nithyanantham\Mini_Projects\results\stock_correlation_matrix.csv"
)

In [ ]:
# 5. Top 5 Gainers and Losers (Month-wise): 
# ● Objective: Provide monthly breakdowns of the top-performing and worst-performing 
# stocks. 
# ● Reason: This analysis will allow users to observe more granular trends and 
# understand which stocks are gaining or losing momentum on a monthly basis. 
# Metrics: 
# ● Group the stock data by month and calculate the monthly return for each stock. 
# ● For each month, identify the top 5 gainers and top 5 losers based on percentage 
# change. 
# ● Create a dashboard-style visualization with 5 charts showing top gainers and losers 
# for each month (12 months total). 
# Visualization: 
# ● Top 5 Gainers and Losers by Month: Create a set of 12 bar charts for each month 
# showing the top 5 gainers and losers based on percentage return. 

In [185]:
sector_df

,Ticker,close,date,high,low,month,open,volume,Daily_Return,Cumulative_return,sector
0,ADANIENT,2464.9500,10/4/2023 5:30,2502.7500,2392.2500,2023-10,2402.2000,2857377,0.0325,0.0325,MINING
1,ADANIENT,2466.3500,10/5/2023 5:30,2486.5000,2446.4000,2023-10,2477.9500,1132455,0.0006,0.0331,MINING
2,ADANIENT,2478.1000,10/6/2023 5:30,2514.9500,2466.0500,2023-10,2466.3500,1510035,0.0048,0.0381,MINING
3,ADANIENT,2442.6000,10/9/2023 5:30,2459.7000,2411.3000,2023-10,2440.0000,1408224,-0.0143,0.0232,MINING
4,ADANIENT,2498.3000,10/10/2023 5:30,2517.9500,2443.0000,2023-10,2443.0000,1771910,0.0228,0.0465,MINING
...,...,...,...,...,...,...,...,...,...,...,...
14145,WIPRO,566.7000,11/14/2024 5:30,574.5500,564.2000,2024-11,568.9500,4891760,-0.0040,0.3977,SOFTWARE
14146,WIPRO,552.8500,11/18/2024 5:30,566.7000,540.3000,2024-11,566.7000,7644882,-0.0244,0.3635,SOFTWARE
14147,WIPRO,562.0000,11/19/2024 5:30,569.8000,554.7000,2024-11,556.0000,6459889,0.0166,0.3861,SOFTWARE
14148,WIPRO,557.1500,11/21/2024 5:30,567.6000,555.3000,2024-11,562.0000,5836304,-0.0086,0.3742,SOFTWARE


In [ ]:
sector_df.groupby('month')['Daily_Return']

In [188]:
monthly_returns = (
    sector_df.sort_values(['Ticker', 'date'])
      .groupby(['month', 'Ticker'])['Daily_Return']
      .apply(lambda x: (1 + x).prod() - 1)
      .reset_index(name='monthly_return')
)

In [189]:
monthly_returns

,month,Ticker,monthly_return
0,2023-10,ADANIENT,-0.0388
1,2023-10,ADANIPORTS,-0.0561
2,2023-10,APOLLOHOSP,-0.0584
3,2023-10,ASIANPAINT,-0.0540
4,2023-10,AXISBANK,-0.0569
...,...,...,...
695,2024-11,TECHM,0.0863
696,2024-11,TITAN,0.0127
697,2024-11,TRENT,-0.0667
698,2024-11,ULTRACEMCO,0.0280


In [190]:
top5_gainers = (
    monthly_returns
    .sort_values(['month', 'monthly_return'], ascending=[True, False])
    .groupby('month')
    .head(5)
)

In [191]:
top5_gainers

,month,Ticker,monthly_return
32,2023-10,NESTLEIND,0.0860
13,2023-10,COALINDIA,0.0766
5,2023-10,BAJAJ-AUTO,0.0593
37,2023-10,SBILIFE,0.0583
47,2023-10,TRENT,0.0464
...,...,...,...
680,2024-11,M&M,0.1042
695,2024-11,TECHM,0.0863
675,2024-11,INFY,0.0825
667,2024-11,HCLTECH,0.0749


In [192]:
top5_losers = (
    monthly_returns
    .sort_values(['month', 'monthly_return'], ascending=[True, True])
    .groupby('month')
    .head(5)
)

In [193]:
top5_losers

,month,Ticker,monthly_return
43,2023-10,TATASTEEL,-0.0723
45,2023-10,TECHM,-0.0683
38,2023-10,SBIN,-0.0620
7,2023-10,BAJFINANCE,-0.0596
2,2023-10,APOLLOHOSP,-0.0584
...,...,...,...
650,2024-11,ADANIENT,-0.2440
651,2024-11,ADANIPORTS,-0.1738
653,2024-11,ASIANPAINT,-0.1579
661,2024-11,BRITANNIA,-0.1534


In [194]:
top5_gainers.to_csv(
    r"C:\Nithyanantham\Mini_Projects\results\top5_gainers_monthwise.csv",
    index=False
)

top5_losers.to_csv(
    r"C:\Nithyanantham\Mini_Projects\results\top5_losers_monthwise.csv",
    index=False
)